In [1]:
!pip install docling boto3 python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 31.0 MB/s eta 0:00:00
   

In [2]:
import os
import time
import re
import shutil
import boto3
from dotenv import load_dotenv
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

In [3]:
# 1. Cargar variables de entorno
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION")

S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
S3_PREFIX_DOCS = os.getenv("S3_PREFIX_DOCS")
S3_PREFIX_BRONCE= os.getenv("S3_PREFIX_BRONCE")

In [4]:
# 2. Inicializar cliente S3
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

In [5]:
def obtener_archivos_s3(prefijo):
    """Devuelve una lista de nombres de archivos en un prefijo dado."""
    archivos = []
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=prefijo):
        if 'Contents' in page:
            for obj in page['Contents']:
                # Evitar agregar el propio directorio como archivo
                if not obj['Key'].endswith('/'):
                    archivos.append(obj['Key'])
    return archivos

In [6]:
def obtener_carpetas_procesadas():
    """Obtiene los nombres de los documentos que ya tienen su carpeta en Bronce."""
    carpetas = set()
    paginator = s3_client.get_paginator('list_objects_v2')
    # Buscamos usando el delimitador '/' para encontrar "subcarpetas"
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=S3_PREFIX_BRONCE, Delimiter='/'):
        if 'CommonPrefixes' in page:
            for prefix in page['CommonPrefixes']:
                # Extraemos solo el nombre de la carpeta (ej. de 'bronze/processed/doc1/' sacamos 'doc1')
                nombre_carpeta = prefix['Prefix'].replace(S3_PREFIX_BRONCE, '').strip('/')
                carpetas.add(nombre_carpeta)
    return carpetas

In [7]:
print("Inicializando motor Docling (Modo Ligero - Sin imágenes)...")
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = False # Apagamos la extracción visual

doc_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

Inicializando motor Docling (Modo Ligero - Sin imágenes)...


In [8]:
def procesar_pipeline_ingesta():
    print("Iniciando Pipeline: Ingesta -> Bronce (Estructura Docling Ligera)")

    pdfs_en_docs = obtener_archivos_s3(S3_PREFIX_DOCS)
    carpetas_en_bronce = obtener_carpetas_procesadas()

    for s3_key in pdfs_en_docs:
        nombre_pdf = os.path.basename(s3_key)
        nombre_base = nombre_pdf.replace('.pdf', '')

        # VERIFICADOR: Si la carpeta ya existe en Bronce, lo saltamos
        if nombre_base in carpetas_en_bronce:
            print(f" Saltando: {nombre_pdf} (Carpeta ya existe en Bronce)")
            continue

        print(f"\n Descargando: {nombre_pdf}...")
        local_pdf_path = f"/content/{nombre_pdf}"
        s3_client.download_file(S3_BUCKET_NAME, s3_key, local_pdf_path)

        print(f" Procesando estructuralmente con Docling: {nombre_pdf}...")
        tiempo_inicio = time.perf_counter()

        try:
            # 1. Extraer con Docling
            resultado = doc_converter.convert(local_pdf_path)

            # 2. Convertir a Markdown
            md_texto = resultado.document.export_to_markdown()

            # 3. Guardar temporalmente en local
            local_md_path = f"/content/{nombre_base}.md"
            with open(local_md_path, 'w', encoding='utf-8') as f:
                f.write(md_texto)

            # 4. Subir a S3 (Mantenemos la lógica de la cápsula/carpeta)
            # Ej: bronze/processed/FDS_1/FDS_1.md
            ruta_s3_destino = f"{S3_PREFIX_BRONCE}{nombre_base}/{nombre_base}.md"
            s3_client.upload_file(local_md_path, S3_BUCKET_NAME, ruta_s3_destino)

            tiempo_fin = time.perf_counter() - tiempo_inicio
            print(f"  Éxito. Cápsula '{nombre_base}/' creada en {tiempo_fin:.2f} seg.")

            # Limpieza del archivo markdown local
            os.remove(local_md_path)

        except Exception as e:
            print(f"  Error de Docling procesando {nombre_pdf}: {e}")

        finally:
            # Limpieza local del PDF para no saturar el disco de Colab
            if os.path.exists(local_pdf_path):
                os.remove(local_pdf_path)

In [9]:
# Ejecutar el flujo
procesar_pipeline_ingesta()

Iniciando Pipeline: Ingesta -> Bronce (Estructura Docling Ligera)
 Saltando: Esmalte Uretano AR Comp. B.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 1 -pintura-antideslizante-para-canchas.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 10 - pintura-epoxi-poliamida-blanco-13243-10012925-10344427-hoja-de-seguridad-pdf.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 11  - Ecosphere Premium.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 12 - PINTURA ANTICORROSIVA GRIS 507 _ 10014333-10012454-10171712 _COL (Versión 3) (1).pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 13 -AVANTEX-FACHADA.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 14 -PINTURA-ACRILICA-PARA-TRÁFICO.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 15 - impra.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 16 - sellador-nitrocelulosico.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 17 - inmunizante-universal.pdf (Carpeta ya existe en Bronce)
 Saltando: FDS 18 - KORAZA BLANCO 2650 _ 10015206-10016171-10017460-

[INFO] 2026-04-01 05:35:10,597 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-01 05:35:10,610 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-01 05:35:10,619 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.7.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-01 05:35:11,467 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-04-01 05:35:11,807 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-01 05:35:11,810 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-01 05:35:13,038 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-01 05:35:13,043 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-01 05:35:13,046 [RapidOCR] download_file.py:68: Initiat

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  Éxito. Cápsula 'FDS 21 -pintura-para-trafico-acrilico-base-solvente-13722-10017277-10015687/' creada en 47.40 seg.

 Descargando: FDS 22 - Esmalte Uretano AR Comp. B.pdf...
 Procesando estructuralmente con Docling: FDS 22 - Esmalte Uretano AR Comp. B.pdf...
  Éxito. Cápsula 'FDS 22 - Esmalte Uretano AR Comp. B/' creada en 5.15 seg.

 Descargando: FDS 3 - ESMALTE-POLIURETANO.pdf...
 Procesando estructuralmente con Docling: FDS 3 - ESMALTE-POLIURETANO.pdf...


  Éxito. Cápsula 'FDS 3 - ESMALTE-POLIURETANO/' creada en 3.80 seg.

 Descargando: FDS 4 -catalizador-duraglass.pdf...
 Procesando estructuralmente con Docling: FDS 4 -catalizador-duraglass.pdf...


  Éxito. Cápsula 'FDS 4 -catalizador-duraglass/' creada en 6.64 seg.

 Descargando: FDS 5 - KORAZA PRO 550 BLANCO  _ 10350789-10289257 _COL (Versión 1).pdf...
 Procesando estructuralmente con Docling: FDS 5 - KORAZA PRO 550 BLANCO  _ 10350789-10289257 _COL (Versión 1).pdf...
  Éxito. Cápsula 'FDS 5 - KORAZA PRO 550 BLANCO  _ 10350789-10289257 _COL (Versión 1)/' creada en 19.61 seg.

 Descargando: FDS 6 - PINTURA ANTICORROSIVA GRIS 507 _ 10014333-10012454-10171712 _COL (Versión 3).pdf...
 Procesando estructuralmente con Docling: FDS 6 - PINTURA ANTICORROSIVA GRIS 507 _ 10014333-10012454-10171712 _COL (Versión 3).pdf...
  Éxito. Cápsula 'FDS 6 - PINTURA ANTICORROSIVA GRIS 507 _ 10014333-10012454-10171712 _COL (Versión 3)/' creada en 27.44 seg.

 Descargando: FDS 7 - PINTUTRAFICO ACRILICO BASE SOLVENTE MOD. BLANCO 13754 - 1 G _ 10115022 _COL (1).pdf...
 Procesando estructuralmente con Docling: FDS 7 - PINTUTRAFICO ACRILICO BASE SOLVENTE MOD. BLANCO 13754 - 1 G _ 10115022 _COL (1).pdf...


  Éxito. Cápsula 'FDS 7 - PINTUTRAFICO ACRILICO BASE SOLVENTE MOD. BLANCO 13754 - 1 G _ 10115022 _COL (1)/' creada en 28.50 seg.

 Descargando: FDS 8 - PINTUTRAFICO ALQUIDICO AMARILLO 659 _ 10012970-10015685 _COL (Versión 2).pdf...
 Procesando estructuralmente con Docling: FDS 8 - PINTUTRAFICO ALQUIDICO AMARILLO 659 _ 10012970-10015685 _COL (Versión 2).pdf...
  Éxito. Cápsula 'FDS 8 - PINTUTRAFICO ALQUIDICO AMARILLO 659 _ 10012970-10015685 _COL (Versión 2)/' creada en 28.67 seg.

 Descargando: FDS 9 - OXIGEL.pdf...
 Procesando estructuralmente con Docling: FDS 9 - OXIGEL.pdf...


  Éxito. Cápsula 'FDS 9 - OXIGEL/' creada en 5.10 seg.
